In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# MY CODE


In [2]:
import os
os.environ["WANDB_PROJECT"] = "23f3000843-t22026"
os.environ["WANDB_ENTITY"] = "varnitchourasiya27-indian-institute-of-technology-madras"

In [3]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType
from torch.utils.data import Dataset
import wandb
from kaggle_secrets import UserSecretsClient
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

options = ['A', 'B', 'C', 'D', 'E']

secrets = UserSecretsClient()
wandb.login(key=secrets.get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: varnitchourasiya27 (varnitchourasiya27-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print("Train shape:", train.shape)
print("Test shape :", test.shape)

Train shape: (2000, 8)
Test shape : (500, 7)


In [5]:
def map_at_3(df, predict_fn):
    scores = []
    for _, row in df.iterrows():
        prediction = predict_fn(row)
        predicted_labels = prediction.split()
        correct = row['answer']
        score = 0.0
        if correct in predicted_labels:
            rank = predicted_labels.index(correct) + 1
            score = 1.0 / rank
        scores.append(score)
    return np.mean(scores)

def evaluate_and_log(model_name, predict_fn, sample_size=200):
    run = wandb.init(
        entity="varnitchourasiya27-indian-institute-of-technology-madras",
        project="23f3000843-t22026",
        name=model_name,
        config={"model": model_name, "sample_size": sample_size}
    )
    sample = train.sample(sample_size, random_state=42)
    score = map_at_3(sample, predict_fn)
    wandb.log({"MAP@3": score})
    print(f"{model_name} → Local MAP@3: {score:.4f}")
    wandb.finish()
    return score

In [6]:
from transformers import AutoTokenizer, AutoModelForMultipleChoice

model_name = "google/electra-base-discriminator"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMultipleChoice.from_pretrained(model_name)

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

ElectraForMultipleChoice LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings_project.weight                 | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
electra.embeddings_project.bias                   | UNEXPECTED | 
sequence_summary.summary.weight                   | MISSING    | 
classifier.bias                                   | MISSING    | 
classifier.weight                                 | MISSING    | 
sequence_summary.summary.bias                     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	

In [7]:
data_files = {"train": "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"}
raw_datasets = load_dataset("csv", data_files=data_files)

Generating train split: 0 examples [00:00, ? examples/s]

In [8]:
def preprocess_multiple_choice(examples):
    first_sentences = [[context] * 5 for context in examples["prompt"]]
    second_sentences = [
        [examples["A"][i], examples["B"][i], examples["C"][i], examples["D"][i], examples["E"][i]]
        for i in range(len(examples["prompt"]))
    ]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized_examples = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=256,
    )

    features = {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized_examples.items()}

    # Map letter answers to integer indices
    label_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
    features["labels"] = [label_map[a] for a in examples["answer"]]

    return features
# Apply it to your dataset
tokenized_datasets = raw_datasets.map(preprocess_multiple_choice, batched=True)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [9]:
from dataclasses import dataclass
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
from typing import Optional, Union
import torch

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0].keys() else "labels"
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        # Flatten again for padding
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features
        ]
        flattened_features = sum(flattened_features, [])
        
        # Pad
        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        # Un-flatten
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch

In [10]:
from transformers import TrainingArguments, Trainer
import numpy as np

# 1. Define the training arguments (Updated for the newest transformers version!)
training_args = TrainingArguments(
    output_dir="./electra_mcq_results",
    eval_strategy="epoch",  # <--- This is the fixed line!
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to="wandb", 
    logging_steps=10
)

# 2. Define the accuracy metric function
def compute_metrics(eval_predictions):
    predictions, label_ids = eval_predictions
    preds = np.argmax(predictions, axis=1)
    return {"accuracy": (preds == label_ids).astype(np.float32).mean().item()}

# 1. Automatically split off 10% of the training data for validation
dataset_splits = tokenized_datasets["train"].train_test_split(test_size=0.1)

# 2. Initialize the Trainer using the new splits
# 2. Initialize the Trainer using the new splits (and processing_class!)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_splits["train"],
    eval_dataset=dataset_splits["test"],  
    processing_class=tokenizer,  # <--- THIS IS THE FIX! (Replaced 'tokenizer')
    data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

In [11]:
# Start the fine-tuning process!
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.388600,0.202056,1.000000
2,0.013487,0.011329,1.000000
3,0.002866,0.000044,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=675, training_loss=0.6106636358059391, metrics={'train_runtime': 350.6084, 'train_samples_per_second': 15.402, 'train_steps_per_second': 1.925, 'total_flos': 1478732520714960.0, 'train_loss': 0.6106636358059391, 'epoch': 3.0})

In [12]:
import torch.nn.functional as F

idx_to_label = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}

def electra_predict_fn(row):
    first_sentences = [row["prompt"]] * 5
    second_sentences = [row["A"], row["B"], row["C"], row["D"], row["E"]]

    inputs = tokenizer(
        first_sentences, second_sentences,
        truncation=True, max_length=256, padding=True,
        return_tensors="pt"
    )
    # reshape to (1, num_choices, seq_len) as the model expects
    inputs = {k: v.unsqueeze(0).to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits.squeeze(0)  # shape: (5,)
    probs = F.softmax(logits, dim=0)

    top3_idx = torch.argsort(probs, descending=True)[:3]
    return " ".join([idx_to_label[i] for i in top3_idx.tolist()])

In [13]:
score = evaluate_and_log("electra-base-mcq-v1", electra_predict_fn, sample_size=500)

eval/accuracy,▁▁▁
eval/loss,█▁▁
eval/runtime,▄▁█
eval/samples_per_second,▄█▁
eval/steps_per_second,▄█▁
train/epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇█████
train/global_step,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
train/grad_norm,▂▃▅▆▅▇▅▇▅▅▄▁█▂█▅█▄▂▆▃▃▃▁▂▁▁▁▁▁▁▃▁▁▂▁▁▁▁▁
train/learning_rate,████▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▁▁
train/loss,█▇▇▆▆▅▄▄▃▃▂▂▁▂▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/accuracy,1


electra-base-mcq-v1 → Local MAP@3: 1.0000


MAP@3,▁
MAP@3,1


In [14]:
def build_submission(test_df, predict_fn, out_path="submission.csv"):
    preds = test_df.apply(predict_fn, axis=1)
    sub = pd.DataFrame({"ID": test_df["id"], "Prediction": preds})  # read lowercase 'id', write as 'ID'
    sub.to_csv(out_path, index=False)
    return sub

build_submission(test, electra_predict_fn)

,ID,Prediction
0,1,A B D
1,2,B D E
2,3,B E C
3,4,E C D
4,5,C B D
...,...,...
495,496,A E D
496,497,C A E
497,498,B D C
498,499,E B D
